# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!git clone https://github.com/Imvixh/flyrank-ml-internship.git /content/flyrank-ml-internship

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.


In [10]:
from pathlib import Path

ROOT = Path("/content/flyrank-ml-internship")

print("Repository exists:", ROOT.exists())
print("Data exists:", (ROOT / "data/raw/content_refresh_anonymized.csv").exists())
print("W06 notebook exists:", (ROOT / "work/notebooks/w06_validation_audit.ipynb").exists())

Repository exists: True
Data exists: True
W06 notebook exists: True


### Finding 1 — AI referrals are measurable but still a limited traffic source

The research paper reports that AI-referred traffic can be observed in search and website analytics data, but the size of this traffic should be interpreted carefully because AI referral sessions are relatively sparse compared with conventional search traffic.

**Methodology question:** How exactly is the AI-referral label defined, and are the measurements based on a consistent set of platforms and time windows? I would want the denominator and inclusion rules to be explicit before treating the reported percentage as representative of all search traffic.

**Validation question:** Does the validation design test whether the observed AI-referral pattern remains stable across clients and time periods, rather than only within the original sample?

### Finding 2 — Search visibility and content characteristics are associated with performance

The paper reports relationships between content/search characteristics and observed performance outcomes.

**Methodology question:** Are these relationships measured using independent observations, or can multiple pages from the same client contribute correlated observations? If pages from the same client are present on both sides of a validation split, performance could look stronger than it generalizes.

**Validation question:** A grouped-by-client or time-aware validation design would make the generalization claim more convincing because it tests whether the relationship survives when the model encounters unseen clients or a later period.

### My overall reading

These findings are useful as descriptive evidence, but they should not be interpreted as proof of causality. For my own ML work, I use the same standard: an observed association can support decision-making, but it does not prove that changing one variable will cause the outcome to change.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [12]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/flyrank-ml-internship")

data_path = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Load the dataset from the cloned repository
ROOT = "/content/flyrank-ml-internship"
data_path = f"{ROOT}/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

# Target: observed declining outcome
df["is_declining"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Same 20 observable features used for the Week-5 model
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

X = df[feature_cols]
y = df["is_declining"]
groups = df["client_id"]

# Grouped split: no client appears in both train and test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("===== ML-09 HONEST GROUPED VALIDATION =====")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Client overlap: {len(train_clients & test_clients)}")
print(f"Features used: {len(feature_cols)}")

# Random Forest — selected Week-5 model
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# Model probabilities
probabilities = model.predict_proba(X_test)[:, 1]

# Precision@50
top50 = np.argsort(probabilities)[::-1][:50]
model_precision50 = y_test.iloc[top50].mean()

# ROC-AUC
model_auc = roc_auc_score(y_test, probabilities)

# Week-4 baseline
test = df.iloc[test_idx].copy()

test["prior_ctr_pct"] = np.where(
    test["impressions_prev_30d"] > 0,
    test["clicks_prev_30d"]
    / test["impressions_prev_30d"] * 100,
    np.nan
)

test["stale"] = test["days_since_last_update"] >= 104
test["visible"] = test["impressions_prev_30d"] >= 300
test["weak_ctr"] = (
    (test["prior_ctr_pct"] < 1.0)
    & test["prior_ctr_pct"].notna()
)

test["baseline_score"] = (
    3 * test["stale"].astype(int)
    + 2 * test["visible"].astype(int)
    + 2 * test["weak_ctr"].astype(int)
)

baseline_top50 = (
    test.sort_values(
        ["baseline_score",
         "impressions_prev_30d",
         "days_since_last_update"],
        ascending=[False, False, False]
    )
    .head(50)
)

baseline_precision50 = baseline_top50["is_declining"].mean()

print("\n===== MODEL VS BASELINE =====")
print(f"Random Forest Precision@50: {model_precision50:.2%}")
print(f"Random Forest ROC-AUC: {model_auc:.2%}")
print(f"Week-4 Baseline Precision@50: {baseline_precision50:.2%}")

comparison = pd.DataFrame({
    "method": [
        "Random Forest",
        "Week-4 Baseline"
    ],
    "Precision@50": [
        model_precision50,
        baseline_precision50
    ]
})

display(comparison)

===== ML-09 HONEST GROUPED VALIDATION =====
Training rows: 23,837
Test rows: 6,163
Training clients: 25
Test clients: 7
Client overlap: 0
Features used: 20

===== MODEL VS BASELINE =====
Random Forest Precision@50: 70.00%
Random Forest ROC-AUC: 61.21%
Week-4 Baseline Precision@50: 30.00%


,method,Precision@50
0,Random Forest,0.7
1,Week-4 Baseline,0.3


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [14]:
# ML-09 — Leakage Audit

# Fields that must NOT be used as model inputs
forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
]

leaked_features = [
    feature for feature in feature_cols
    if feature in forbidden_fields
]

print("===== ML-09 LEAKAGE AUDIT =====")

print("Model features checked:")
for feature in feature_cols:
    print(" -", feature)

print("\nForbidden outcome/current-window fields:")
for field in forbidden_fields:
    print(" -", field)

print("\nPotential leakage in model features:", leaked_features)

if len(leaked_features) == 0:
    print("\nRESULT: PASS")
    print(
        "No outcome or current-window performance fields "
        "are included in the model feature set."
    )
else:
    print("\nRESULT: FAIL")
    print("Potential leakage detected:", leaked_features)

===== ML-09 LEAKAGE AUDIT =====
Model features checked:
 - search_volume
 - competition
 - cpc
 - word_count
 - char_count
 - impressions_90d
 - clicks_90d
 - pageviews_90d
 - sessions_90d
 - users_90d
 - engaged_sessions_90d
 - ai_sessions_90d
 - scroll_events_90d
 - days_with_impressions
 - days_with_sessions
 - content_age_days
 - age_tier_order
 - days_since_last_update
 - ctr
 - avg_position

Forbidden outcome/current-window fields:
 - trend_direction
 - trend_pct
 - is_declining
 - impressions_last_30d
 - clicks_last_30d
 - sessions_last_30d

Potential leakage in model features: []

RESULT: PASS
No outcome or current-window performance fields are included in the model feature set.


## 4. Claim rewrite

### Original claim

The Random Forest model performs better than the Week-4 baseline and can identify declining pages.

### Safer rewritten claim

On the grouped-by-client evaluation split, the Random Forest achieved higher Precision@50 than the Week-4 baseline in this experiment.

This is measured evidence on this dataset and validation split. It does not prove that the same performance will hold for future data or unseen conditions.

The model should therefore be treated as a decision-support tool for prioritizing pages for human review. It does not establish causality, prove Google's ranking algorithm, or guarantee that a page will decline.

The grouped-by-client split provides a stricter validation design because clients in the test set were not used for model training.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-check

- [x] Section 1 is complete: two research-paper findings and methodology/validation questions are documented.
- [x] Section 2 is complete: the model was evaluated using an honest grouped-by-client split.
- [x] The Random Forest was compared with the Week-4 baseline using Precision@50.
- [x] Section 3 is complete: the model feature set was checked for leakage.
- [x] Outcome fields and current-window performance fields were excluded from the model inputs.
- [x] Section 4 is complete: the original claim was rewritten using careful, measured language.
- [x] Results are described as observed evidence and decision-support, not causal proof.
- [x] The notebook uses no client names, private queries, credentials, or sensitive information.
- [x] The notebook runs from top to bottom without errors.
- [x] The validation results are based on the actual dataset and held-out evaluation split.
- [x] The completed notebook is ready to be committed to my personal GitHub repository.